In [2]:
import pandas as pd 


In [61]:
df = pd.read_csv(r"D:\pypipeline\data\raw\static\hourly\aqi\multi\shankapark_hourly.csv")

## Multi data contains PM2.5, PM10, PM 1 and tsp

With one year limited data, but has more features. Hence using this data for modeling.
source: https://opendatanepal.com/datasets/realtime-air-quality-datasets

- PM2.5(µg/m3)
- PM10(µg/m3)
- PM1(µg/m3)

The below steps extract all the relevant concetration and make them into columns

In [62]:
df_pm25 = df[df["particulate_matter "]=="PM2.5"].drop(columns=["particulate_matter ","station"]).rename(columns={"value":"pm25_value"})
df_pm10 = df[df["particulate_matter "]=="PM10"].drop(columns=["particulate_matter ","station"]).rename(columns={"value":"pm10_value"})
df_pm1 = df[df["particulate_matter "]=="PM1"].drop(columns=["particulate_matter ","station"]).rename(columns={"value":"pm1_value"})


Outer merge to reatain all the data in features

In [63]:
df_merged = df_pm25.merge(df_pm10, on='datetime', how='outer') \
                   .merge(df_pm1, on='datetime', how='outer') \
                   

In [64]:
df_merged["datetime"] = pd.to_datetime(df_merged["datetime"], utc=True, infer_datetime_format=True)
df_merged["datetime"] = df_merged["datetime"].dt.tz_convert("Asia/Kathmandu")

C:\Users\ZENBOOK\AppData\Local\Temp\ipykernel_19440\1992135032.py:1: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_merged["datetime"] = pd.to_datetime(df_merged["datetime"], utc=True, infer_datetime_format=True)


In [67]:
df_merged

,datetime,pm25_value,pm10_value,pm1_value
0,2019-02-11 16:17:00+05:45,28.4,34.9,25.3
1,2019-02-11 17:02:00+05:45,22.7,24.3,21.2
2,2019-02-11 17:12:00+05:45,21.6,NaN,NaN
3,2019-02-11 17:52:00+05:45,19.4,21.4,18.3
4,2019-02-11 18:02:00+05:45,19.3,20.4,18.3
...,...,...,...,...
80520,2020-05-09 21:29:41+05:45,NaN,36.8,NaN
80521,2020-05-09 21:29:42+05:45,NaN,NaN,28.1
80522,2020-05-09 21:39:28+05:45,NaN,NaN,29.3
80523,2020-05-09 21:39:46+05:45,33.7,NaN,NaN


In [66]:
df_merged.isna().sum()

datetime          0
pm25_value    45666
pm10_value    45109
pm1_value     46874
dtype: int64

Resample data into hourly frequency by taking mean of values in each hour

In [68]:
num_cols = ['pm25_value', 'pm10_value', 'pm1_value']
df_merged[num_cols] = df_merged[num_cols].apply(pd.to_numeric, errors='coerce')

df_hourly = df_merged.sort_values(by='datetime').resample('H', on='datetime').mean().reset_index().round(2)
df_hourly.set_index('datetime', inplace=True)


C:\Users\ZENBOOK\AppData\Local\Temp\ipykernel_19440\3273166718.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df_merged.sort_values(by='datetime').resample('H', on='datetime').mean().reset_index().round(2)


In [70]:
df_hourly.sort_values(by=df_hourly.index.name)


,pm25_value,pm10_value,pm1_value
datetime,,,
2019-02-11 16:00:00+05:45,28.40,34.90,25.30
2019-02-11 17:00:00+05:45,21.23,22.85,19.75
2019-02-11 18:00:00+05:45,19.30,20.40,18.30
2019-02-11 19:00:00+05:45,16.80,17.50,16.17
2019-02-11 20:00:00+05:45,18.60,22.20,17.00
...,...,...,...
2020-05-09 17:00:00+05:45,20.85,26.90,19.55
2020-05-09 18:00:00+05:45,23.70,35.41,21.39
2020-05-09 19:00:00+05:45,29.53,39.57,26.93


## Feature Engineering

In [74]:
df_hourly['is_missing'] = df_hourly.isna().any(axis=1).astype(int)


In [76]:
df_hourly['gap_group'] = (df_hourly['is_missing'] != df_hourly['is_missing'].shift()).cumsum()
gap_summary = (
    df_hourly[df_hourly['is_missing'] == 1]
    .groupby('gap_group')
    .size()
    .reset_index(name='gap_length')
)



In [77]:
gap_summary.max()

gap_group     580
gap_length    478
dtype: int64

In [80]:
df_hourly[df_hourly["gap_group"]==362]

,pm25_value,pm10_value,pm1_value,is_missing,gap_group
datetime,,,,,
2019-10-16 13:00:00+05:45,NaN,NaN,NaN,1,362
2019-10-16 14:00:00+05:45,NaN,NaN,NaN,1,362
2019-10-16 15:00:00+05:45,NaN,NaN,NaN,1,362
2019-10-16 16:00:00+05:45,NaN,NaN,NaN,1,362
2019-10-16 17:00:00+05:45,NaN,NaN,NaN,1,362
...,...,...,...,...,...
2019-11-05 06:00:00+05:45,NaN,NaN,NaN,1,362
2019-11-05 07:00:00+05:45,NaN,NaN,NaN,1,362
2019-11-05 08:00:00+05:45,NaN,NaN,NaN,1,362


In [71]:
df_featured_enginnered = df_hourly.copy()


Week and night features

In [72]:
df_featured_enginnered["hour"] = df_featured_enginnered.index.hour
df_featured_enginnered["day_of_week"] = df_featured_enginnered.index.dayofweek
df_featured_enginnered["is_weekend"] = df_featured_enginnered["day_of_week"].isin([5,6]).astype(int)
df_featured_enginnered["is_night"]=df_featured_enginnered["hour"].isin([0,1,2,3,4,5]).astype(int)

In [73]:
df_featured_enginnered

,pm25_value,pm10_value,pm1_value,hour,day_of_week,is_weekend,is_night
datetime,,,,,,,
2019-02-11 16:00:00+05:45,28.40,34.90,25.30,16,0,0,0
2019-02-11 17:00:00+05:45,21.23,22.85,19.75,17,0,0,0
2019-02-11 18:00:00+05:45,19.30,20.40,18.30,18,0,0,0
2019-02-11 19:00:00+05:45,16.80,17.50,16.17,19,0,0,0
2019-02-11 20:00:00+05:45,18.60,22.20,17.00,20,0,0,0
...,...,...,...,...,...,...,...
2020-05-09 17:00:00+05:45,20.85,26.90,19.55,17,5,1,0
2020-05-09 18:00:00+05:45,23.70,35.41,21.39,18,5,1,0
2020-05-09 19:00:00+05:45,29.53,39.57,26.93,19,5,1,0


### Time lags
using 24 hour lags for daily forcasting


In [ ]:
lags = [i for i in range(1,25)]
for lag in lags:
    df["pm25_value_lag {lag}"] = df_featured_enginnered["pm25_value"].shift(lag)
    df["pm10_value_lag {lag}"] = df_featured_enginnered["pm10_value"].shift(lag)
    df["pm1_value_lag {lag}"] = df_featured_enginnered["pm1_value"].shift(lag)


[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24]

,pm25_value,pm10_value,pm1_value,tsp_value,hour,day_of_week,is_weekend,is_night
datetime,,,,,,,,
2019-01-01 00:00:00+05:45,34.30,37.20,32.10,37.20,0,1,0,1
2019-01-01 01:00:00+05:45,42.75,56.60,33.30,84.95,1,1,0,1
2019-01-01 02:00:00+05:45,33.05,42.05,27.80,45.10,2,1,0,1
2019-01-01 03:00:00+05:45,32.50,35.85,28.60,36.20,3,1,0,1
2019-01-01 04:00:00+05:45,34.45,37.20,32.75,37.30,4,1,0,1
...,...,...,...,...,...,...,...,...
2020-05-09 17:00:00+05:45,15.13,26.77,13.73,30.43,17,5,1,0
2020-05-09 18:00:00+05:45,19.20,36.10,15.95,181.40,18,5,1,0
2020-05-09 19:00:00+05:45,20.10,26.00,17.24,26.00,19,5,1,0
